# Comprehensive Tree Model Analysis for CMAPSS RUL Prediction

This notebook tests various tree-based models across all four CMAPSS datasets (FD001-FD004) for Remaining Useful Life (RUL) prediction.

## Models Tested:
1. **Single Decision Tree** - Baseline
2. **Bagging Models**:
   - Random Forest
   - Extra Trees
3. **Gradient Boosting Models**:
   - HistGradientBoosting (scikit-learn)
   - XGBoost
   - LightGBM
   - CatBoost
4. **Era-Split Models**:
   - EraHistGradientBoosting

## Analysis Approach:
- Individual dataset analysis (FD001, FD002, FD003, FD004)
- Cross-dataset analysis (combined data)
- Era-based validation (engine units as eras)


## 1. Imports and Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(1337)

# Sklearn imports
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

# Gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Era boosting
from erasplit.ensemble import EraHistGradientBoostingRegressor

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All imports successful")

## 2. Data Loading Functions

In [ ]:
# Column definitions
index_names = ['unit_nr', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names = ['s_{}'.format(i) for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

# Constant features to drop (based on FD001 analysis)
constant_features = ['s_1', 's_5', 's_10', 's_16', 's_18', 's_19', 'setting_3']

def load_cmapss_dataset(dataset_name, data_dir='../data'):
    """
    Load a single CMAPSS dataset (e.g., 'FD001', 'FD002', etc.)
    
    Returns:
        train_df: Training dataframe
        test_df: Test dataframe
        RUL_df: True RUL values for test set
    """
    print(f"Loading {dataset_name}...")
    
    # Load data files
    train_path = os.path.join(data_dir, f'train_{dataset_name}.txt')
    test_path = os.path.join(data_dir, f'test_{dataset_name}.txt')
    rul_path = os.path.join(data_dir, f'RUL_{dataset_name}.txt')
    
    train_df = pd.read_csv(train_path, sep='\\s+', header=None, names=col_names)
    test_df = pd.read_csv(test_path, sep='\\s+', header=None, names=col_names)
    RUL_df = pd.read_csv(rul_path, sep='\\s+', header=None, names=['RUL'])
    
    # Drop constant features
    train_df.drop(constant_features, axis=1, inplace=True, errors='ignore')
    test_df.drop(constant_features, axis=1, inplace=True, errors='ignore')
    
    print(f"  Train shape: {train_df.shape}")
    print(f"  Test shape: {test_df.shape}")
    print(f"  Number of train units: {train_df['unit_nr'].nunique()}")
    print(f"  Number of test units: {test_df['unit_nr'].nunique()}")
    
    return train_df, test_df, RUL_df

def smooth_sensor_data(df, window_size=3):
    """
    Apply moving average smoothing to sensor data.
    Smoothing is done per unit to avoid mixing data from different engines.
    
    Args:
        df: Dataframe with sensor data
        window_size: Window size for moving average (default: 3)
    
    Returns:
        Smoothed dataframe
    """
    smoothed_df = df.copy()
    
    # Identify sensor and setting columns to smooth
    cols_to_smooth = [col for col in df.columns 
                     if col.startswith('s_') or col.startswith('setting_')]
    
    # Smooth each unit separately
    for unit_nr in df['unit_nr'].unique():
        unit_mask = smoothed_df['unit_nr'] == unit_nr
        
        # Apply rolling mean to sensor columns for this unit
        for col in cols_to_smooth:
            smoothed_df.loc[unit_mask, col] = (
                smoothed_df.loc[unit_mask, col]
                .rolling(window=window_size, min_periods=1, center=True)
                .mean()
            )
    
    return smoothed_df

def engineer_features(train_df, test_df, smooth=False, window_size=3):
    """
    Add engineered features:
    - max_cycles: Maximum cycles for each unit
    - TTF: Time to Failure (max_cycles - time_cycles)
    - FTTF: Fractional Time to Failure (TTF / max_cycles)
    
    Args:
        train_df: Training dataframe
        test_df: Test dataframe
        smooth: Whether to apply smoothing (default: False)
        window_size: Window size for smoothing if smooth=True (default: 3)
    """
    # Apply smoothing if requested
    if smooth:
        print(f"  Applying moving average smoothing (window={window_size})...")
        train_df = smooth_sensor_data(train_df, window_size)
        test_df = smooth_sensor_data(test_df, window_size)
    
    # Get max cycles for each unit
    train_df = pd.merge(
        train_df, 
        train_df.groupby('unit_nr', as_index=False)['time_cycles'].max(),
        how='left', 
        on='unit_nr'
    )
    train_df.rename(columns={'time_cycles_x': 'time_cycles', 'time_cycles_y': 'max_cycles'}, inplace=True)
    
    test_df = pd.merge(
        test_df, 
        test_df.groupby('unit_nr', as_index=False)['time_cycles'].max(),
        how='left', 
        on='unit_nr'
    )
    test_df.rename(columns={'time_cycles_x': 'time_cycles', 'time_cycles_y': 'max_cycles'}, inplace=True)
    
    # Add TTF (Time to Failure)
    train_df['TTF'] = train_df['max_cycles'] - train_df['time_cycles']
    test_df['TTF'] = test_df['max_cycles'] - test_df['time_cycles']
    
    return train_df, test_df

def scale_features(train_df, test_df):
    """
    Scale features using MinMaxScaler and add FTTF (Fractional TTF)
    """
    # Identify feature columns (exclude unit_nr, time_cycles, max_cycles, TTF)
    feature_cols = [col for col in train_df.columns 
                   if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF']]
    
    scaler = MinMaxScaler()
    
    scaled_train_df = train_df.copy()
    scaled_train_df[feature_cols] = scaler.fit_transform(scaled_train_df[feature_cols])
    
    scaled_test_df = test_df.copy()
    scaled_test_df[feature_cols] = scaler.transform(scaled_test_df[feature_cols])
    
    # Add fractional TTF
    scaled_train_df['FTTF'] = scaled_train_df['TTF'] / scaled_train_df['max_cycles']
    scaled_test_df['FTTF'] = scaled_test_df['TTF'] / scaled_test_df['max_cycles']
    
    return scaled_train_df, scaled_test_df

print("✓ Data loading functions defined")

## 3. Model Evaluation Functions

In [ ]:
def transform_predictions_to_ttf(y_pred_fttf, y_test_fttf, test_df):
    """
    Transform fractional TTF predictions back to TTF scale
    """
    test_max_cycles = test_df.groupby('unit_nr')['max_cycles'].first()
    
    Y_predict_TTF = []
    Y_test_TTF = []
    
    for unit_nr in test_df['unit_nr'].unique():
        unit_mask = test_df['unit_nr'] == unit_nr
        max_cycles = test_max_cycles[unit_nr]
        
        # Get predictions and actual values for this unit
        unit_predictions = y_pred_fttf[unit_mask]
        unit_actual = y_test_fttf[unit_mask]
        
        # Transform back to TTF scale
        Y_predict_TTF.extend(unit_predictions * max_cycles)
        Y_test_TTF.extend(unit_actual * max_cycles)
    
    return np.array(Y_predict_TTF), np.array(Y_test_TTF)

def evaluate_model(model, X_train, Y_train, X_test, Y_test, model_name, test_df, train_df=None, verbose=True):
    """
    Train model and return comprehensive evaluation metrics
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Training {model_name}")
        print(f"{'='*60}")
    
    # Training
    start_time = time.time()
    if model_name == "EraHistGradientBoosting":
        if train_df is None:
            raise ValueError("train_df with 'unit_nr' is required for Era Boosting model.")
        eras = train_df['unit_nr'].to_numpy()
        model.fit(X_train, Y_train, eras)
    else:
        model.fit(X_train, Y_train)
    training_time = time.time() - start_time
    
    # Predictions
    start_time = time.time()
    Y_pred_train = model.predict(X_train)
    Y_pred_test = model.predict(X_test)
    inference_time = time.time() - start_time
    
    # FTTF scale metrics
    train_rmse_fttf = np.sqrt(mean_squared_error(Y_train, Y_pred_train))
    test_rmse_fttf = np.sqrt(mean_squared_error(Y_test, Y_pred_test))
    test_mae_fttf = mean_absolute_error(Y_test, Y_pred_test)
    test_r2_fttf = r2_score(Y_test, Y_pred_test)
    
    # TTF scale metrics
    Y_pred_test_ttf, Y_test_ttf = transform_predictions_to_ttf(Y_pred_test, Y_test, test_df)
    test_rmse_ttf = np.sqrt(mean_squared_error(Y_test_ttf, Y_pred_test_ttf))
    test_mae_ttf = mean_absolute_error(Y_test_ttf, Y_pred_test_ttf)
    
    if verbose:
        print(f"Training Time: {training_time:.3f} seconds")
        print(f"Inference Time: {inference_time:.3f} seconds")
        print(f"\nFTTF Scale Metrics:")
        print(f"  Train RMSE: {train_rmse_fttf:.6f}")
        print(f"  Test RMSE:  {test_rmse_fttf:.6f}")
        print(f"  Test MAE:   {test_mae_fttf:.6f}")
        print(f"  Test R²:    {test_r2_fttf:.6f}")
        print(f"\nTTF Scale Metrics:")
        print(f"  Test RMSE:  {test_rmse_ttf:.2f} cycles")
        print(f"  Test MAE:   {test_mae_ttf:.2f} cycles")
    
    return {
        'model': model,
        'model_name': model_name,
        'training_time': training_time,
        'inference_time': inference_time,
        'train_rmse_fttf': train_rmse_fttf,
        'test_rmse_fttf': test_rmse_fttf,
        'test_mae_fttf': test_mae_fttf,
        'test_r2_fttf': test_r2_fttf,
        'test_rmse_ttf': test_rmse_ttf,
        'test_mae_ttf': test_mae_ttf,
        'Y_pred_test': Y_pred_test,
        'Y_pred_test_ttf': Y_pred_test_ttf,
        'Y_test_ttf': Y_test_ttf
    }

print("✓ Evaluation functions defined")

## 4. Model Definitions

In [ ]:
def get_tree_models():
    """
    Define all tree-based models to test
    """
    models = {
        # 1. Baseline Decision Tree
        'DecisionTree': DecisionTreeRegressor(
            max_depth=10,
            min_samples_split=10,
            random_state=42
        ),
        
        # 2. Bagging Models
        'RandomForest': RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        ),
        
        'ExtraTrees': ExtraTreesRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        ),
        
        # 3. Gradient Boosting Models
        'HistGradientBoosting': HistGradientBoostingRegressor(
            max_iter=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42
        ),
        
        'XGBoost': xgb.XGBRegressor(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        ),
        
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        
        'CatBoost': CatBoostRegressor(
            iterations=100,
            depth=5,
            learning_rate=0.1,
            random_state=42,
            verbose=0
        ),
        
        # 4. Era-based Model
        'EraHistGradientBoosting': EraHistGradientBoostingRegressor(
            early_stopping=False,
            n_jobs=-1,
            colsample_bytree=1,
            max_bins=5,
            max_depth=5,
            max_leaf_nodes=16,
            min_samples_leaf=16,
            max_iter=100,
            l2_regularization=0.1,
            learning_rate=0.01,
            blama=1,
            min_agreement_threshold=0,
            verbose=0
        )
    }
    
    return models

print("✓ Model definitions created")

## 5. Visualization Functions

In [ ]:
def plot_model_comparison(results_df, dataset_name):
    """
    Create comprehensive comparison plots for all models
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Sort by test RMSE for consistent ordering
    results_df = results_df.sort_values('Test RMSE (TTF)')
    
    # 1. RMSE comparison
    ax1 = axes[0, 0]
    x_pos = np.arange(len(results_df))
    ax1.bar(x_pos, results_df['Test RMSE (TTF)'], alpha=0.7, color='steelblue')
    ax1.set_title(f'{dataset_name}: Test RMSE (TTF Scale)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Models')
    ax1.set_ylabel('RMSE (cycles)')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. R² comparison
    ax2 = axes[0, 1]
    ax2.bar(x_pos, results_df['Test R²'], alpha=0.7, color='coral')
    ax2.set_title(f'{dataset_name}: Test R² Score', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Models')
    ax2.set_ylabel('R² Score')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(results_df['Model'], rotation=45, ha='right')
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Training time comparison
    ax3 = axes[1, 0]
    ax3.bar(x_pos, results_df['Training Time (s)'], alpha=0.7, color='seagreen')
    ax3.set_title(f'{dataset_name}: Training Time', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Models')
    ax3.set_ylabel('Time (seconds)')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(results_df['Model'], rotation=45, ha='right')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Overfitting analysis
    ax4 = axes[1, 1]
    ax4.scatter(results_df['Train RMSE (FTTF)'], results_df['Test RMSE (FTTF)'], 
               alpha=0.7, s=100, color='purple')
    for i, model in enumerate(results_df['Model']):
        ax4.annotate(model, 
                    (results_df['Train RMSE (FTTF)'].iloc[i], results_df['Test RMSE (FTTF)'].iloc[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    max_val = max(results_df['Train RMSE (FTTF)'].max(), results_df['Test RMSE (FTTF)'].max())
    ax4.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='Perfect fit')
    ax4.set_title(f'{dataset_name}: Overfitting Analysis', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Train RMSE (FTTF)')
    ax4.set_ylabel('Test RMSE (FTTF)')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_predictions_vs_actual(results, dataset_name, n_units=4):
    """
    Plot predictions vs actual for best performing model
    """
    # Find best model by RMSE
    best_model_name = min(results.items(), key=lambda x: x[1]['test_rmse_ttf'])[0]
    best_result = results[best_model_name]
    
    print(f"\n📊 Best Model: {best_model_name}")
    print(f"   Test RMSE: {best_result['test_rmse_ttf']:.2f} cycles")
    print(f"   Test R²: {best_result['test_r2_fttf']:.4f}")
    
    # Get test dataframe from results (stored during evaluation)
    # For now, we'll skip the time series plot since we don't have test_df accessible here
    # Instead, create a scatter plot
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    Y_test_ttf = best_result['Y_test_ttf']
    Y_pred_ttf = best_result['Y_pred_test_ttf']
    
    ax.scatter(Y_test_ttf, Y_pred_ttf, alpha=0.5, s=20)
    
    # Perfect prediction line
    min_val = min(Y_test_ttf.min(), Y_pred_ttf.min())
    max_val = max(Y_test_ttf.max(), Y_pred_ttf.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, alpha=0.8, label='Perfect prediction')
    
    ax.set_xlabel('Actual TTF (cycles)', fontsize=12)
    ax.set_ylabel('Predicted TTF (cycles)', fontsize=12)
    ax.set_title(f'{dataset_name}: {best_model_name} - Predictions vs Actual', 
                fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Add metrics text
    correlation = np.corrcoef(Y_test_ttf, Y_pred_ttf)[0, 1]
    metrics_text = f'RMSE: {best_result["test_rmse_ttf"]:.2f} cycles\n'
    metrics_text += f'MAE: {best_result["test_mae_ttf"]:.2f} cycles\n'
    metrics_text += f'R²: {best_result["test_r2_fttf"]:.4f}\n'
    metrics_text += f'Correlation: {correlation:.4f}'
    
    ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
           verticalalignment='top', fontsize=10)
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization functions defined")

## 6. Load All Datasets

In [ ]:
# Load all four CMAPSS datasets
datasets = {}
datasets_smoothed = {}

for dataset_name in ['FD001', 'FD002', 'FD003', 'FD004']:
    # Load unsmoothed data
    train_df, test_df, RUL_df = load_cmapss_dataset(dataset_name)
    
    # Engineer features (unsmoothed)
    train_df_unsmooth, test_df_unsmooth = engineer_features(train_df.copy(), test_df.copy(), smooth=False)
    
    # Engineer features (smoothed with window=3)
    train_df_smooth, test_df_smooth = engineer_features(train_df.copy(), test_df.copy(), smooth=True, window_size=3)
    
    # Scale features (unsmoothed)
    scaled_train_df, scaled_test_df = scale_features(train_df_unsmooth, test_df_unsmooth)
    
    # Scale features (smoothed)
    scaled_train_df_smooth, scaled_test_df_smooth = scale_features(train_df_smooth, test_df_smooth)
    
    # Store unsmoothed data
    datasets[dataset_name] = {
        'train': train_df_unsmooth,
        'test': test_df_unsmooth,
        'RUL': RUL_df,
        'scaled_train': scaled_train_df,
        'scaled_test': scaled_test_df
    }
    
    # Store smoothed data
    datasets_smoothed[dataset_name] = {
        'train': train_df_smooth,
        'test': test_df_smooth,
        'RUL': RUL_df,
        'scaled_train': scaled_train_df_smooth,
        'scaled_test': scaled_test_df_smooth
    }
    print()

print("✓ All datasets loaded and preprocessed (both smoothed and unsmoothed versions)")

## 7. Individual Dataset Analysis

### 7.1 FD001 Analysis

In [ ]:
print("\n" + "="*80)
print("FD001 DATASET ANALYSIS")
print("="*80)

dataset_name = 'FD001'
data = datasets[dataset_name]

# Prepare data
feature_cols = [col for col in data['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]

X_train = data['scaled_train'][feature_cols]
Y_train = data['scaled_train']['FTTF']
X_test = data['scaled_test'][feature_cols]
Y_test = data['scaled_test']['FTTF']

print(f"\nFeature shape: {X_train.shape}")
print(f"Number of features: {len(feature_cols)}")

# Get models
models = get_tree_models()

# Evaluate all models
results_fd001 = {}
for model_name, model in models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test'], data['scaled_train']
        )
    else:
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test']
        )
    results_fd001[model_name] = result

# Create results dataframe
results_df_fd001 = pd.DataFrame([{
    'Model': result['model_name'],
    'Train RMSE (FTTF)': result['train_rmse_fttf'],
    'Test RMSE (FTTF)': result['test_rmse_fttf'],
    'Test RMSE (TTF)': result['test_rmse_ttf'],
    'Test MAE (TTF)': result['test_mae_ttf'],
    'Test R²': result['test_r2_fttf'],
    'Training Time (s)': result['training_time'],
    'Inference Time (s)': result['inference_time']
} for result in results_fd001.values()])

results_df_fd001 = results_df_fd001.sort_values('Test RMSE (TTF)')
print("\n" + "="*80)
print("FD001 RESULTS SUMMARY")
print("="*80)
print(results_df_fd001.to_string(index=False))

# Visualizations
plot_model_comparison(results_df_fd001, 'FD001')
plot_predictions_vs_actual(results_fd001, 'FD001')

### 7.2 FD002 Analysis

In [ ]:
print("\n" + "="*80)
print("FD002 DATASET ANALYSIS")
print("="*80)

dataset_name = 'FD002'
data = datasets[dataset_name]

# Prepare data
feature_cols = [col for col in data['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]

X_train = data['scaled_train'][feature_cols]
Y_train = data['scaled_train']['FTTF']
X_test = data['scaled_test'][feature_cols]
Y_test = data['scaled_test']['FTTF']

# Get models
models = get_tree_models()

# Evaluate all models
results_fd002 = {}
for model_name, model in models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test'], data['scaled_train']
        )
    else:
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test']
        )
    results_fd002[model_name] = result

# Create results dataframe
results_df_fd002 = pd.DataFrame([{
    'Model': result['model_name'],
    'Train RMSE (FTTF)': result['train_rmse_fttf'],
    'Test RMSE (FTTF)': result['test_rmse_fttf'],
    'Test RMSE (TTF)': result['test_rmse_ttf'],
    'Test MAE (TTF)': result['test_mae_ttf'],
    'Test R²': result['test_r2_fttf'],
    'Training Time (s)': result['training_time'],
    'Inference Time (s)': result['inference_time']
} for result in results_fd002.values()])

results_df_fd002 = results_df_fd002.sort_values('Test RMSE (TTF)')
print("\n" + "="*80)
print("FD002 RESULTS SUMMARY")
print("="*80)
print(results_df_fd002.to_string(index=False))

# Visualizations
plot_model_comparison(results_df_fd002, 'FD002')
plot_predictions_vs_actual(results_fd002, 'FD002')

### 7.3 FD003 Analysis

In [ ]:
print("\n" + "="*80)
print("FD003 DATASET ANALYSIS")
print("="*80)

dataset_name = 'FD003'
data = datasets[dataset_name]

# Prepare data
feature_cols = [col for col in data['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]

X_train = data['scaled_train'][feature_cols]
Y_train = data['scaled_train']['FTTF']
X_test = data['scaled_test'][feature_cols]
Y_test = data['scaled_test']['FTTF']

# Get models
models = get_tree_models()

# Evaluate all models
results_fd003 = {}
for model_name, model in models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test'], data['scaled_train']
        )
    else:
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test']
        )
    results_fd003[model_name] = result

# Create results dataframe
results_df_fd003 = pd.DataFrame([{
    'Model': result['model_name'],
    'Train RMSE (FTTF)': result['train_rmse_fttf'],
    'Test RMSE (FTTF)': result['test_rmse_fttf'],
    'Test RMSE (TTF)': result['test_rmse_ttf'],
    'Test MAE (TTF)': result['test_mae_ttf'],
    'Test R²': result['test_r2_fttf'],
    'Training Time (s)': result['training_time'],
    'Inference Time (s)': result['inference_time']
} for result in results_fd003.values()])

results_df_fd003 = results_df_fd003.sort_values('Test RMSE (TTF)')
print("\n" + "="*80)
print("FD003 RESULTS SUMMARY")
print("="*80)
print(results_df_fd003.to_string(index=False))

# Visualizations
plot_model_comparison(results_df_fd003, 'FD003')
plot_predictions_vs_actual(results_fd003, 'FD003')

### 7.4 FD004 Analysis

In [ ]:
print("\n" + "="*80)
print("FD004 DATASET ANALYSIS")
print("="*80)

dataset_name = 'FD004'
data = datasets[dataset_name]

# Prepare data
feature_cols = [col for col in data['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]

X_train = data['scaled_train'][feature_cols]
Y_train = data['scaled_train']['FTTF']
X_test = data['scaled_test'][feature_cols]
Y_test = data['scaled_test']['FTTF']

# Get models
models = get_tree_models()

# Evaluate all models
results_fd004 = {}
for model_name, model in models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test'], data['scaled_train']
        )
    else:
        result = evaluate_model(
            model, X_train, Y_train, X_test, Y_test, 
            model_name, data['scaled_test']
        )
    results_fd004[model_name] = result

# Create results dataframe
results_df_fd004 = pd.DataFrame([{
    'Model': result['model_name'],
    'Train RMSE (FTTF)': result['train_rmse_fttf'],
    'Test RMSE (FTTF)': result['test_rmse_fttf'],
    'Test RMSE (TTF)': result['test_rmse_ttf'],
    'Test MAE (TTF)': result['test_mae_ttf'],
    'Test R²': result['test_r2_fttf'],
    'Training Time (s)': result['training_time'],
    'Inference Time (s)': result['inference_time']
} for result in results_fd004.values()])

results_df_fd004 = results_df_fd004.sort_values('Test RMSE (TTF)')
print("\n" + "="*80)
print("FD004 RESULTS SUMMARY")
print("="*80)
print(results_df_fd004.to_string(index=False))

# Visualizations
plot_model_comparison(results_df_fd004, 'FD004')
plot_predictions_vs_actual(results_fd004, 'FD004')

## 8. Cross-Dataset Comparison

In [ ]:
print("\n" + "="*80)
print("CROSS-DATASET PERFORMANCE COMPARISON")
print("="*80)

# Combine all results
all_results = {
    'FD001': results_df_fd001,
    'FD002': results_df_fd002,
    'FD003': results_df_fd003,
    'FD004': results_df_fd004
}

# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

models_list = results_df_fd001['Model'].tolist()
datasets_list = ['FD001', 'FD002', 'FD003', 'FD004']

# 1. RMSE comparison across datasets
ax1 = axes[0, 0]
x = np.arange(len(models_list))
width = 0.2
for i, dataset in enumerate(datasets_list):
    rmse_values = [all_results[dataset][all_results[dataset]['Model'] == model]['Test RMSE (TTF)'].values[0] 
                  for model in models_list]
    ax1.bar(x + i*width, rmse_values, width, label=dataset, alpha=0.8)
ax1.set_xlabel('Models')
ax1.set_ylabel('Test RMSE (cycles)')
ax1.set_title('Test RMSE Comparison Across Datasets', fontweight='bold')
ax1.set_xticks(x + width*1.5)
ax1.set_xticklabels(models_list, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. R² comparison across datasets
ax2 = axes[0, 1]
for i, dataset in enumerate(datasets_list):
    r2_values = [all_results[dataset][all_results[dataset]['Model'] == model]['Test R²'].values[0] 
                for model in models_list]
    ax2.bar(x + i*width, r2_values, width, label=dataset, alpha=0.8)
ax2.set_xlabel('Models')
ax2.set_ylabel('Test R²')
ax2.set_title('Test R² Comparison Across Datasets', fontweight='bold')
ax2.set_xticks(x + width*1.5)
ax2.set_xticklabels(models_list, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. Average performance ranking
ax3 = axes[1, 0]
avg_rmse = {}
for model in models_list:
    rmse_values = [all_results[dataset][all_results[dataset]['Model'] == model]['Test RMSE (TTF)'].values[0] 
                  for dataset in datasets_list]
    avg_rmse[model] = np.mean(rmse_values)

sorted_models = sorted(avg_rmse.items(), key=lambda x: x[1])
models_sorted = [m[0] for m in sorted_models]
rmse_sorted = [m[1] for m in sorted_models]

ax3.barh(models_sorted, rmse_sorted, alpha=0.7, color='steelblue')
ax3.set_xlabel('Average Test RMSE (cycles)')
ax3.set_title('Average Model Performance Across All Datasets', fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# 4. Training time comparison
ax4 = axes[1, 1]
for i, dataset in enumerate(datasets_list):
    time_values = [all_results[dataset][all_results[dataset]['Model'] == model]['Training Time (s)'].values[0] 
                  for model in models_list]
    ax4.bar(x + i*width, time_values, width, label=dataset, alpha=0.8)
ax4.set_xlabel('Models')
ax4.set_ylabel('Training Time (seconds)')
ax4.set_title('Training Time Comparison Across Datasets', fontweight='bold')
ax4.set_xticks(x + width*1.5)
ax4.set_xticklabels(models_list, rotation=45, ha='right')
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary table
print("\nAverage Performance Ranking (by RMSE):")
print("-" * 60)
for i, (model, rmse) in enumerate(sorted_models, 1):
    print(f"{i}. {model:30s} - Avg RMSE: {rmse:6.2f} cycles")

## 9. Combined Dataset Analysis

Train on all datasets combined to see if models can generalize across different operating conditions.

In [ ]:
print("\n" + "="*80)
print("COMBINED DATASET ANALYSIS (ALL FD001-FD004)")
print("="*80)

# Combine all training and test data
combined_train_dfs = []
combined_test_dfs = []

for dataset_name in ['FD001', 'FD002', 'FD003', 'FD004']:
    data = datasets[dataset_name]
    
    # Add dataset identifier and offset unit numbers to avoid conflicts
    train_df = data['scaled_train'].copy()
    test_df = data['scaled_test'].copy()
    
    # Offset unit numbers by dataset (e.g., FD001: 0-99, FD002: 1000-1259, etc.)
    offset = int(dataset_name[-1]) * 1000
    train_df['unit_nr'] = train_df['unit_nr'] + offset
    test_df['unit_nr'] = test_df['unit_nr'] + offset
    
    train_df['dataset'] = dataset_name
    test_df['dataset'] = dataset_name
    
    combined_train_dfs.append(train_df)
    combined_test_dfs.append(test_df)

combined_train = pd.concat(combined_train_dfs, ignore_index=True)
combined_test = pd.concat(combined_test_dfs, ignore_index=True)

print(f"\nCombined training samples: {len(combined_train)}")
print(f"Combined test samples: {len(combined_test)}")
print(f"Total units (train): {combined_train['unit_nr'].nunique()}")
print(f"Total units (test): {combined_test['unit_nr'].nunique()}")

# Prepare data
feature_cols = [col for col in combined_train.columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF', 'dataset']]

X_train_combined = combined_train[feature_cols]
Y_train_combined = combined_train['FTTF']
X_test_combined = combined_test[feature_cols]
Y_test_combined = combined_test['FTTF']

# Get models
models = get_tree_models()

# Evaluate all models
results_combined = {}
for model_name, model in models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train_combined, Y_train_combined, X_test_combined, Y_test_combined,
            model_name, combined_test, combined_train
        )
    else:
        result = evaluate_model(
            model, X_train_combined, Y_train_combined, X_test_combined, Y_test_combined,
            model_name, combined_test
        )
    results_combined[model_name] = result

# Create results dataframe
results_df_combined = pd.DataFrame([{
    'Model': result['model_name'],
    'Train RMSE (FTTF)': result['train_rmse_fttf'],
    'Test RMSE (FTTF)': result['test_rmse_fttf'],
    'Test RMSE (TTF)': result['test_rmse_ttf'],
    'Test MAE (TTF)': result['test_mae_ttf'],
    'Test R²': result['test_r2_fttf'],
    'Training Time (s)': result['training_time'],
    'Inference Time (s)': result['inference_time']
} for result in results_combined.values()])

results_df_combined = results_df_combined.sort_values('Test RMSE (TTF)')
print("\n" + "="*80)
print("COMBINED DATASET RESULTS SUMMARY")
print("="*80)
print(results_df_combined.to_string(index=False))

# Visualizations
plot_model_comparison(results_df_combined, 'COMBINED (FD001-FD004)')
plot_predictions_vs_actual(results_combined, 'COMBINED (FD001-FD004)')

## 10. Final Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY: BEST MODELS PER DATASET")
print("="*80)

summary_data = []

# Individual datasets
for dataset_name, results_df in [('FD001', results_df_fd001), 
                                  ('FD002', results_df_fd002),
                                  ('FD003', results_df_fd003),
                                  ('FD004', results_df_fd004)]:
    best_row = results_df.iloc[0]
    summary_data.append({
        'Dataset': dataset_name,
        'Best Model': best_row['Model'],
        'Test RMSE (TTF)': best_row['Test RMSE (TTF)'],
        'Test R²': best_row['Test R²'],
        'Training Time (s)': best_row['Training Time (s)']
    })

# Combined dataset
best_combined = results_df_combined.iloc[0]
summary_data.append({
    'Dataset': 'COMBINED',
    'Best Model': best_combined['Model'],
    'Test RMSE (TTF)': best_combined['Test RMSE (TTF)'],
    'Test R²': best_combined['Test R²'],
    'Training Time (s)': best_combined['Training Time (s)']
})

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print("""
1. Model Performance:
   - Compare RMSE across datasets to identify best overall model
   - Consider R² scores for goodness of fit
   - Look for consistency across different datasets

2. Era-Based Models:
   - EraHistGradientBoosting performance shows if era-awareness helps
   - Compare against standard gradient boosting methods

3. Computational Efficiency:
   - Training time varies significantly between models
   - Consider tradeoff between accuracy and training cost

4. Dataset Characteristics:
   - FD001/FD003: Single operating condition
   - FD002/FD004: Multiple operating conditions
   - Performance differences highlight model robustness

5. Next Steps:
   - Hyperparameter tuning for best models
   - Feature importance analysis
   - Cross-dataset generalization testing
   - Ensemble methods combining top models
""")

print("\n✓ Analysis Complete!")

In [ ]:
print("\n" + "="*80)
print("GRID SEARCH FINAL SUMMARY")
print("="*80)

# Create summary dataframe
summary_data = []

# Add baseline results
if best_model_tuned_results:
    summary_data.append({
        'Model': f'{best_model_name} (Baseline)',
        'Test RMSE (TTF)': results_df_fd001[results_df_fd001['Model'] == best_model_name]['Test RMSE (TTF)'].values[0],
        'Test R²': results_df_fd001[results_df_fd001['Model'] == best_model_name]['Test R²'].values[0],
        'Status': 'Baseline'
    })
    
    summary_data.append({
        'Model': best_model_tuned_results['model_name'],
        'Test RMSE (TTF)': best_model_tuned_results['test_rmse_ttf'],
        'Test R²': best_model_tuned_results['test_r2'],
        'Status': 'Tuned'
    })

summary_data.append({
    'Model': 'EraHistGradientBoosting (Baseline)',
    'Test RMSE (TTF)': results_df_fd001[results_df_fd001['Model'] == 'EraHistGradientBoosting']['Test RMSE (TTF)'].values[0],
    'Test R²': results_df_fd001[results_df_fd001['Model'] == 'EraHistGradientBoosting']['Test R²'].values[0],
    'Status': 'Baseline'
})

summary_data.append({
    'Model': era_model_tuned_results['model_name'],
    'Test RMSE (TTF)': era_model_tuned_results['test_rmse_ttf'],
    'Test R²': era_model_tuned_results['test_r2'],
    'Status': 'Tuned'
})

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE comparison
ax1 = axes[0]
baseline_mask = summary_df['Status'] == 'Baseline'
tuned_mask = summary_df['Status'] == 'Tuned'

x_pos = np.arange(len(summary_df) // 2)
width = 0.35

baseline_rmse = summary_df[baseline_mask]['Test RMSE (TTF)'].values
tuned_rmse = summary_df[tuned_mask]['Test RMSE (TTF)'].values
model_names = [m.split(' (')[0] for m in summary_df[baseline_mask]['Model'].values]

ax1.bar(x_pos - width/2, baseline_rmse, width, label='Baseline', alpha=0.8, color='steelblue')
ax1.bar(x_pos + width/2, tuned_rmse, width, label='Tuned', alpha=0.8, color='coral')
ax1.set_xlabel('Models')
ax1.set_ylabel('Test RMSE (cycles)')
ax1.set_title('Grid Search: Baseline vs Tuned Performance', fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(model_names, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Improvement visualization
ax2 = axes[1]
improvements = []
model_labels = []

if best_model_tuned_results:
    improvements.append(best_model_tuned_results['improvement'])
    model_labels.append(best_model_name)

improvements.append(era_model_tuned_results['improvement'])
model_labels.append('EraHistGradientBoosting')

colors = ['green' if x > 0 else 'red' for x in improvements]
ax2.barh(model_labels, improvements, alpha=0.7, color=colors)
ax2.set_xlabel('RMSE Improvement (cycles)')
ax2.set_title('RMSE Improvement from Grid Search\n(Positive = Better)', fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.grid(axis='x', alpha=0.3)

# Add improvement text
for i, (model, improvement) in enumerate(zip(model_labels, improvements)):
    ax2.text(improvement, i, f'  {improvement:.2f}', 
            va='center', ha='left' if improvement > 0 else 'right')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY TAKEAWAYS FROM GRID SEARCH")
print("="*80)

if best_model_tuned_results:
    print(f"\n1. {best_model_name}:")
    print(f"   - Improvement: {best_model_tuned_results['improvement']:.2f} cycles")
    print(f"   - Search time: {best_model_tuned_results['search_time']:.1f} seconds")
    print(f"   - Best params: {best_model_tuned_results['best_params']}")

print(f"\n2. EraHistGradientBoosting:")
print(f"   - Improvement: {era_model_tuned_results['improvement']:.2f} cycles")
print(f"   - Search time: {era_model_tuned_results['search_time']:.1f} seconds")
print(f"   - Best params: {era_model_tuned_results['best_params']}")

print("\n" + "="*80)
print("✓ Grid Search Analysis Complete!")
print("="*80)

### 12.3 Grid Search Summary

Compare tuned models with baseline performance.

In [ ]:
print(f"\n{'='*60}")
print(f"GRID SEARCH: EraHistGradientBoosting")
print(f"{'='*60}")

# Parameter grid for Era Boosting
era_param_grid = {
    'max_iter': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_leaf_nodes': [10, 16, 31],
    'min_samples_leaf': [10, 16, 25],
    'l2_regularization': [0.01, 0.1, 0.5],
    'blama': [0.5, 1.0, 2.0],
    'min_agreement_threshold': [0, 0.25, 0.5]
}

print(f"\nParameter grid:")
for param, values in era_param_grid.items():
    print(f"  {param}: {values}")

print(f"\nPerforming 3-fold cross-validation grid search...")
print("(This may take several minutes due to era-aware training...)")

# Get eras (unit numbers) for training
eras_train = data['scaled_train']['unit_nr'].to_numpy()

# Custom wrapper for EraHistGradientBoosting to work with GridSearchCV
class EraBoostingWrapper:
    def __init__(self, max_iter=100, max_depth=5, learning_rate=0.01, 
                 max_leaf_nodes=16, min_samples_leaf=16, l2_regularization=0.1,
                 blama=1, min_agreement_threshold=0):
        self.max_iter = max_iter
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.max_leaf_nodes = max_leaf_nodes
        self.min_samples_leaf = min_samples_leaf
        self.l2_regularization = l2_regularization
        self.blama = blama
        self.min_agreement_threshold = min_agreement_threshold
        self.model = None
        
    def fit(self, X, y, eras=None):
        if eras is None:
            # Generate dummy eras if not provided (for CV splits)
            eras = np.zeros(len(X))
        
        self.model = EraHistGradientBoostingRegressor(
            early_stopping=False,
            n_jobs=-1,
            colsample_bytree=1,
            max_bins=5,
            max_iter=self.max_iter,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            max_leaf_nodes=self.max_leaf_nodes,
            min_samples_leaf=self.min_samples_leaf,
            l2_regularization=self.l2_regularization,
            blama=self.blama,
            min_agreement_threshold=self.min_agreement_threshold,
            verbose=0
        )
        self.model.fit(X, y, eras)
        return self
    
    def predict(self, X):
        return self.model.predict(X)
    
    def get_params(self, deep=True):
        return {
            'max_iter': self.max_iter,
            'max_depth': self.max_depth,
            'learning_rate': self.learning_rate,
            'max_leaf_nodes': self.max_leaf_nodes,
            'min_samples_leaf': self.min_samples_leaf,
            'l2_regularization': self.l2_regularization,
            'blama': self.blama,
            'min_agreement_threshold': self.min_agreement_threshold
        }
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# Custom CV that preserves eras
from sklearn.model_selection import KFold

class EraAwareCV:
    def __init__(self, n_splits=3):
        self.n_splits = n_splits
        
    def split(self, X, y=None, groups=None):
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        return kf.split(X)
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

start_time = time.time()

# Create wrapper model
era_model_wrapper = EraBoostingWrapper()

# Perform grid search
era_grid_search = GridSearchCV(
    era_model_wrapper,
    era_param_grid,
    scoring=rmse_score,
    cv=EraAwareCV(n_splits=3),
    n_jobs=1,  # Era boosting handles parallelism internally
    verbose=1
)

# Fit with eras
era_grid_search.fit(X_train, Y_train, eras=eras_train)
era_search_time = time.time() - start_time

print(f"\n✓ Grid search completed in {era_search_time:.1f} seconds")
print(f"\nBest parameters found:")
for param, value in era_grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV score (negative RMSE): {era_grid_search.best_score_:.6f}")

# Evaluate best model on test set
best_era_model = era_grid_search.best_estimator_
Y_pred_test_era = best_era_model.predict(X_test)

# FTTF metrics
test_rmse_fttf_era = np.sqrt(mean_squared_error(Y_test, Y_pred_test_era))
test_r2_era = r2_score(Y_test, Y_pred_test_era)

# TTF metrics
Y_pred_test_ttf_era, Y_test_ttf_era = transform_predictions_to_ttf(Y_pred_test_era, Y_test, data['scaled_test'])
test_rmse_ttf_era = np.sqrt(mean_squared_error(Y_test_ttf_era, Y_pred_test_ttf_era))
test_mae_ttf_era = mean_absolute_error(Y_test_ttf_era, Y_pred_test_ttf_era)

print(f"\n{'='*60}")
print(f"TUNED ERA BOOSTING PERFORMANCE ON TEST SET")
print(f"{'='*60}")
print(f"Test RMSE (TTF): {test_rmse_ttf_era:.2f} cycles")
print(f"Test MAE (TTF):  {test_mae_ttf_era:.2f} cycles")
print(f"Test R²:         {test_r2_era:.6f}")

# Compare with baseline
baseline_era_rmse = results_df_fd001[results_df_fd001['Model'] == 'EraHistGradientBoosting']['Test RMSE (TTF)'].values[0]
era_improvement = baseline_era_rmse - test_rmse_ttf_era
era_improvement_pct = (era_improvement / baseline_era_rmse) * 100

print(f"\n{'='*60}")
print(f"IMPROVEMENT OVER BASELINE")
print(f"{'='*60}")
print(f"Baseline RMSE: {baseline_era_rmse:.2f} cycles")
print(f"Tuned RMSE:    {test_rmse_ttf_era:.2f} cycles")
print(f"Improvement:   {era_improvement:.2f} cycles ({era_improvement_pct:+.2f}%)")

# Store results
era_model_tuned_results = {
    'model': best_era_model,
    'model_name': 'EraHistGradientBoosting (Tuned)',
    'best_params': era_grid_search.best_params_,
    'test_rmse_ttf': test_rmse_ttf_era,
    'test_mae_ttf': test_mae_ttf_era,
    'test_r2': test_r2_era,
    'improvement': era_improvement,
    'search_time': era_search_time
}

### 12.2 Grid Search for EraHistGradientBoosting

Perform grid search on the EraHistGradientBoosting model with era-aware splits.

In [ ]:
# Define parameter grids for different models
param_grids = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2', None]
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, -1],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [31, 50, 100],
        'subsample': [0.6, 0.8, 1.0]
    },
    'HistGradientBoosting': {
        'max_iter': [100, 200, 300],
        'max_depth': [3, 5, 7, None],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_leaf_nodes': [15, 31, 50, None]
    },
    'CatBoost': {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5]
    }
}

# Function to get base model
def get_base_model(model_name):
    models_dict = {
        'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
        'XGBoost': xgb.XGBRegressor(random_state=42, n_jobs=-1),
        'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
        'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0),
        'ExtraTrees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
        'DecisionTree': DecisionTreeRegressor(random_state=42)
    }
    return models_dict.get(model_name)

# Perform grid search on best model
if best_model_name in param_grids:
    print(f"\n{'='*60}")
    print(f"GRID SEARCH: {best_model_name}")
    print(f"{'='*60}")
    
    base_model = get_base_model(best_model_name)
    param_grid = param_grids[best_model_name]
    
    print(f"\nParameter grid:")
    for param, values in param_grid.items():
        print(f"  {param}: {values}")
    
    print(f"\nPerforming 3-fold cross-validation grid search...")
    print("(This may take several minutes...)")
    
    start_time = time.time()
    grid_search = GridSearchCV(
        base_model,
        param_grid,
        scoring=rmse_score,
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, Y_train)
    search_time = time.time() - start_time
    
    print(f"\n✓ Grid search completed in {search_time:.1f} seconds")
    print(f"\nBest parameters found:")
    for param, value in grid_search.best_params_.items():
        print(f"  {param}: {value}")
    
    print(f"\nBest CV score (negative RMSE): {grid_search.best_score_:.6f}")
    
    # Evaluate best model on test set
    best_model_tuned = grid_search.best_estimator_
    Y_pred_test = best_model_tuned.predict(X_test)
    
    # FTTF metrics
    test_rmse_fttf = np.sqrt(mean_squared_error(Y_test, Y_pred_test))
    test_r2 = r2_score(Y_test, Y_pred_test)
    
    # TTF metrics
    Y_pred_test_ttf, Y_test_ttf = transform_predictions_to_ttf(Y_pred_test, Y_test, data['scaled_test'])
    test_rmse_ttf = np.sqrt(mean_squared_error(Y_test_ttf, Y_pred_test_ttf))
    test_mae_ttf = mean_absolute_error(Y_test_ttf, Y_pred_test_ttf)
    
    print(f"\n{'='*60}")
    print(f"TUNED MODEL PERFORMANCE ON TEST SET")
    print(f"{'='*60}")
    print(f"Test RMSE (TTF): {test_rmse_ttf:.2f} cycles")
    print(f"Test MAE (TTF):  {test_mae_ttf:.2f} cycles")
    print(f"Test R²:         {test_r2:.6f}")
    
    # Compare with baseline
    baseline_rmse = results_df_fd001[results_df_fd001['Model'] == best_model_name]['Test RMSE (TTF)'].values[0]
    improvement = baseline_rmse - test_rmse_ttf
    improvement_pct = (improvement / baseline_rmse) * 100
    
    print(f"\n{'='*60}")
    print(f"IMPROVEMENT OVER BASELINE")
    print(f"{'='*60}")
    print(f"Baseline RMSE: {baseline_rmse:.2f} cycles")
    print(f"Tuned RMSE:    {test_rmse_ttf:.2f} cycles")
    print(f"Improvement:   {improvement:.2f} cycles ({improvement_pct:+.2f}%)")
    
    # Store results
    best_model_tuned_results = {
        'model': best_model_tuned,
        'model_name': f'{best_model_name} (Tuned)',
        'best_params': grid_search.best_params_,
        'test_rmse_ttf': test_rmse_ttf,
        'test_mae_ttf': test_mae_ttf,
        'test_r2': test_r2,
        'improvement': improvement,
        'search_time': search_time
    }
    
else:
    print(f"\n⚠️  Grid search not configured for {best_model_name}")
    print("Skipping grid search for this model.")
    best_model_tuned_results = None

### 12.1 Grid Search for Best Model

Perform grid search on the best performing model from the baseline analysis.

In [ ]:
print("\n" + "="*80)
print("HYPERPARAMETER GRID SEARCH (FD001)")
print("="*80)

# Use FD001 for grid search
dataset_name = 'FD001'
data = datasets[dataset_name]

feature_cols = [col for col in data['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]
X_train = data['scaled_train'][feature_cols]
Y_train = data['scaled_train']['FTTF']
X_test = data['scaled_test'][feature_cols]
Y_test = data['scaled_test']['FTTF']

# Identify best model from FD001 results
best_model_name = results_df_fd001.iloc[0]['Model']
print(f"\n🎯 Best Model from FD001 Analysis: {best_model_name}")
print(f"   Baseline Test RMSE: {results_df_fd001.iloc[0]['Test RMSE (TTF)']:.2f} cycles")

# Custom scorer for RMSE (negative because GridSearchCV maximizes)
from sklearn.metrics import make_scorer

def rmse_scorer(y_true, y_pred):
    return -np.sqrt(mean_squared_error(y_true, y_pred))

rmse_score = make_scorer(rmse_scorer, greater_is_better=True)

## 12. Hyperparameter Grid Search

Perform grid search on:
1. The best performing model (lowest test RMSE)
2. EraHistGradientBoosting model

This will optimize hyperparameters to potentially improve performance further.

In [ ]:
print("\n" + "="*80)
print("SMOOTHED VS UNSMOOTHED DATA COMPARISON (FD001)")
print("="*80)

# Use FD001 for comparison
dataset_name = 'FD001'

# Unsmoothed data
data_unsmooth = datasets[dataset_name]
feature_cols = [col for col in data_unsmooth['scaled_train'].columns 
               if col not in ['unit_nr', 'time_cycles', 'max_cycles', 'TTF', 'FTTF']]
X_train_unsmooth = data_unsmooth['scaled_train'][feature_cols]
Y_train_unsmooth = data_unsmooth['scaled_train']['FTTF']
X_test_unsmooth = data_unsmooth['scaled_test'][feature_cols]
Y_test_unsmooth = data_unsmooth['scaled_test']['FTTF']

# Smoothed data
data_smooth = datasets_smoothed[dataset_name]
X_train_smooth = data_smooth['scaled_train'][feature_cols]
Y_train_smooth = data_smooth['scaled_train']['FTTF']
X_test_smooth = data_smooth['scaled_test'][feature_cols]
Y_test_smooth = data_smooth['scaled_test']['FTTF']

print(f"\n📊 Testing key models on smoothed vs unsmoothed data...")

# Test subset of models for comparison
comparison_models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),
    'EraHistGradientBoosting': EraHistGradientBoostingRegressor(
        early_stopping=False, n_jobs=-1, max_depth=5, max_iter=100, 
        learning_rate=0.01, blama=1, verbose=0
    )
}

results_comparison = {'Unsmoothed': {}, 'Smoothed': {}}

# Evaluate on unsmoothed data
print("\n--- UNSMOOTHED DATA ---")
for model_name, model in comparison_models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train_unsmooth, Y_train_unsmooth, X_test_unsmooth, Y_test_unsmooth,
            model_name, data_unsmooth['scaled_test'], data_unsmooth['scaled_train'], verbose=False
        )
    else:
        result = evaluate_model(
            model, X_train_unsmooth, Y_train_unsmooth, X_test_unsmooth, Y_test_unsmooth,
            model_name, data_unsmooth['scaled_test'], verbose=False
        )
    results_comparison['Unsmoothed'][model_name] = result
    print(f"{model_name:30s} - Test RMSE: {result['test_rmse_ttf']:6.2f} cycles, R²: {result['test_r2_fttf']:.4f}")

# Evaluate on smoothed data  
print("\n--- SMOOTHED DATA (window=3) ---")
for model_name, model in comparison_models.items():
    if model_name == 'EraHistGradientBoosting':
        result = evaluate_model(
            model, X_train_smooth, Y_train_smooth, X_test_smooth, Y_test_smooth,
            model_name, data_smooth['scaled_test'], data_smooth['scaled_train'], verbose=False
        )
    else:
        result = evaluate_model(
            model, X_train_smooth, Y_train_smooth, X_test_smooth, Y_test_smooth,
            model_name, data_smooth['scaled_test'], verbose=False
        )
    results_comparison['Smoothed'][model_name] = result
    print(f"{model_name:30s} - Test RMSE: {result['test_rmse_ttf']:6.2f} cycles, R²: {result['test_r2_fttf']:.4f}")

# Create comparison dataframe
comparison_data = []
for model_name in comparison_models.keys():
    unsmooth_result = results_comparison['Unsmoothed'][model_name]
    smooth_result = results_comparison['Smoothed'][model_name]
    
    comparison_data.append({
        'Model': model_name,
        'Unsmoothed RMSE': unsmooth_result['test_rmse_ttf'],
        'Smoothed RMSE': smooth_result['test_rmse_ttf'],
        'RMSE Improvement': unsmooth_result['test_rmse_ttf'] - smooth_result['test_rmse_ttf'],
        'Unsmoothed R²': unsmooth_result['test_r2_fttf'],
        'Smoothed R²': smooth_result['test_r2_fttf'],
        'R² Improvement': smooth_result['test_r2_fttf'] - unsmooth_result['test_r2_fttf']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("SMOOTHING EFFECT SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE comparison
ax1 = axes[0]
x = np.arange(len(comparison_models))
width = 0.35
unsmooth_rmse = [comparison_df[comparison_df['Model'] == m]['Unsmoothed RMSE'].values[0] 
                 for m in comparison_models.keys()]
smooth_rmse = [comparison_df[comparison_df['Model'] == m]['Smoothed RMSE'].values[0] 
               for m in comparison_models.keys()]

ax1.bar(x - width/2, unsmooth_rmse, width, label='Unsmoothed', alpha=0.8, color='steelblue')
ax1.bar(x + width/2, smooth_rmse, width, label='Smoothed (window=3)', alpha=0.8, color='coral')
ax1.set_xlabel('Models')
ax1.set_ylabel('Test RMSE (cycles)')
ax1.set_title('RMSE: Smoothed vs Unsmoothed Data', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(comparison_models.keys(), rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Improvement bar chart
ax2 = axes[1]
improvements = comparison_df['RMSE Improvement'].values
colors = ['green' if x > 0 else 'red' for x in improvements]
ax2.barh(list(comparison_models.keys()), improvements, alpha=0.7, color=colors)
ax2.set_xlabel('RMSE Improvement (cycles)')
ax2.set_title('RMSE Improvement with Smoothing\n(Positive = Better)', fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Smoothing comparison complete!")

## 11. Smoothed vs Unsmoothed Data Comparison

Compare model performance on smoothed (moving average window=3) vs unsmoothed sensor data.